# Module 06: GoF Design Patterns Scalable Systems — Interactive Laboratory

Every cell below runs the module's **real** implementation from
`project_solution/notification_engine.py`. Nothing here prints a claim it has not verified.

What you will do:

1. Load the engine and inspect what it actually exports.
2. Run its primary workflow and check the assertions that define correctness.
3. **Commit to a prediction**, then run the cell that tests it.
4. Measure a property rather than asserting one.
5. Fix a deliberately broken cell in place.

> The code in cells 4, 6 and 8 is lifted from this module's own test suite, so it
> cannot drift from the implementation. If the API changes, those tests fail
> first and this notebook is regenerated from them.


## 1. Load the engine and introspect it

Rather than trusting a hardcoded list of class names, ask the module what it
actually contains.


In [ ]:
import inspect
import sys
from pathlib import Path

sys.path.insert(0, str(Path('.').resolve() / 'project_solution'))
import notification_engine

classes = [n for n, o in inspect.getmembers(notification_engine, inspect.isclass)
           if o.__module__ == 'notification_engine']
functions = [n for n, o in inspect.getmembers(notification_engine, inspect.isfunction)
             if o.__module__ == 'notification_engine']

print('module   : notification_engine')
print(f'classes  : {classes}')
print(f'functions: {functions}')
print()
for name in classes:
    obj = getattr(notification_engine, name)
    try:
        sig = inspect.signature(obj.__init__)
        params = [p for p in sig.parameters if p != 'self']
    except (TypeError, ValueError):
        params = ['<builtin>']
    print(f'  {name}({", ".join(params)})')

## 2. Baseline: Factory pattern creates proper strategies

This is the module's own `test_factory_pattern_creates_proper_strategies` — real instantiation, real calls, real
assertions. If it runs clean, the property it encodes holds.


In [ ]:
import pytest
from notification_engine import (
    AuditLogDecorator,
    ChannelFactory,
    EmailChannel,
    NotificationMessage,
    PushNotificationChannel,
    RetryDecorator,
    SMSChannel,
)

email = ChannelFactory.create_channel("EMAIL")
assert isinstance(email, EmailChannel)
assert email.channel_name == "EMAIL"

sms = ChannelFactory.create_channel("SMS")
assert isinstance(sms, SMSChannel)
assert sms.channel_name == "SMS"

push = ChannelFactory.create_channel("PUSH", token_store={"token_123"})
assert isinstance(push, PushNotificationChannel)
assert push.channel_name == "PUSH"

with pytest.raises(ValueError, match="Unknown notification channel"):
    ChannelFactory.create_channel("CARRIER_PIGEON")

print('PASSED: test_factory_pattern_creates_proper_strategies')

## 3. 🔮 Prediction — commit before you run

Predict what happens when two subscribers to the same event both raise an exception. Does the third subscriber still receive the event?

Write your answer down. An uncommitted guess teaches nothing, because you will
retro-fit it to whatever the next cell prints.

The next cell runs `test_decorator_pattern_audit_logging`, which tests exactly this property.


In [ ]:
logs: list[str] = []
base_channel = EmailChannel()
audited = AuditLogDecorator(base_channel, audit_log=logs)

msg = NotificationMessage(message_id="msg-1", recipient="dev@example.com", content="System Alert")
res = audited.send(msg)

assert res is True
assert len(logs) == 1
assert "EMAIL" in logs[0]
assert "SUCCESS" in logs[0]
assert "dev@example.com" in logs[0]

print('PASSED: test_decorator_pattern_audit_logging')

## 4. Measure it: Decorator pattern retry on failure

An assertion tells you a property holds. A measurement tells you *how much*.
This cell runs `test_decorator_pattern_retry_on_failure` and times it.


In [ ]:
import time

_t0 = time.perf_counter()

calls = {"count": 0}

class FlakyEmail(EmailChannel):
    def send(self, message: NotificationMessage) -> bool:
        calls["count"] += 1
        if calls["count"] < 2:
            raise ConnectionError("Temporary SMTP timeout")
        return True

retried = RetryDecorator(FlakyEmail(), max_retries=3)
msg = NotificationMessage(message_id="msg-2", recipient="user@test.org", content="Hello")
assert retried.send(msg) is True
assert calls["count"] == 2

_elapsed = (time.perf_counter() - _t0) * 1000
print('PASSED: test_decorator_pattern_retry_on_failure')
print(f'wall clock: {_elapsed:.2f} ms')

## 5. 🛠️ Fix this cell — it is deliberately broken

The cell below asserts something **false** about the real object. Read the
failure, work out the true value from the module's actual behaviour, and correct
the expected number.

Do not delete the assertion. The point is to make it pass by knowing the answer.


In [ ]:
# DELIBERATELY BROKEN - fix the expected value below.
# Hint: print the real value first, then decide what the assertion should say.

exports = [n for n in dir(notification_engine) if not n.startswith('_')]
print(f'actual export count: {len(exports)}')
print(f'actual exports     : {exports}')

EXPECTED_EXPORT_COUNT = 999      # <-- wrong on purpose. Replace it.

assert len(exports) == EXPECTED_EXPORT_COUNT, (
    f'expected {EXPECTED_EXPORT_COUNT} exports, found {len(exports)}. '
    'Read the printed value above and correct the constant.'
)
print('Fixed - assertion now reflects reality.')

### 🎓 Key takeaways

1. Patterns are names for shapes you already needed - not a menu to shop from.
2. One subscriber's failure must not deny the event to the others.
3. The right pattern makes the next change small; the wrong one makes it large.

---

**Continue with this module:**

- [README.md](README.md) — the mental model and failure modes
- [PROJECT_GUIDE.md](PROJECT_GUIDE.md) — build it yourself, in 3 tiers
- [starter/](starter/) — your stubs; run the tests from there to grade yourself
- [debug_lab/SYMPTOMS.md](debug_lab/SYMPTOMS.md) — diagnose planted bugs from the symptom
- [TROUBLESHOOTING_AND_EDGE_CASES.md](TROUBLESHOOTING_AND_EDGE_CASES.md) — real errors, real causes
- [SELF_ASSESSMENT_AND_CHALLENGES.md](SELF_ASSESSMENT_AND_CHALLENGES.md) — quiz and diagnostics
